# Data Cleaning and Preprocessing

**Objective:** Transform raw loan applicant data into a clean, verified dataset ready for feature engineering.


### 1: Import Libraries and Load Data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)

# Load raw training dataset
TRAIN_PATH = Path("../data/train.csv") if Path("../data/train.csv").exists() else Path("data/train.csv")
df = pd.read_csv(TRAIN_PATH)

print(f"Raw dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head(3)

Raw dataset loaded: 233,154 rows x 41 columns


,UniqueID,disbursed_amount,asset_cost,ltv,branch_id,supplier_id,manufacturer_id,Current_pincode_ID,Date.of.Birth,Employment.Type,DisbursalDate,State_ID,Employee_code_ID,MobileNo_Avl_Flag,Aadhar_flag,PAN_flag,VoterID_flag,Driving_flag,Passport_flag,PERFORM_CNS.SCORE,PERFORM_CNS.SCORE.DESCRIPTION,PRI.NO.OF.ACCTS,PRI.ACTIVE.ACCTS,PRI.OVERDUE.ACCTS,PRI.CURRENT.BALANCE,PRI.SANCTIONED.AMOUNT,PRI.DISBURSED.AMOUNT,SEC.NO.OF.ACCTS,SEC.ACTIVE.ACCTS,SEC.OVERDUE.ACCTS,SEC.CURRENT.BALANCE,SEC.SANCTIONED.AMOUNT,SEC.DISBURSED.AMOUNT,PRIMARY.INSTAL.AMT,SEC.INSTAL.AMT,NEW.ACCTS.IN.LAST.SIX.MONTHS,DELINQUENT.ACCTS.IN.LAST.SIX.MONTHS,AVERAGE.ACCT.AGE,CREDIT.HISTORY.LENGTH,NO.OF_INQUIRIES,loan_default
0,420825,50578,58400,89.55,67,22807,45,1441,01-01-84,Salaried,03-08-18,6,1998,1,1,0,0,0,0,0,No Bureau History Available,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0yrs 0mon,0yrs 0mon,0,0
1,537409,47145,65550,73.23,67,22807,45,1502,31-07-85,Self employed,26-09-18,6,1998,1,1,0,0,0,0,598,I-Medium Risk,1,1,1,27600,50200,50200,0,0,0,0,0,0,1991,0,0,1,1yrs 11mon,1yrs 11mon,0,1
2,417566,53278,61360,89.63,67,22807,45,1497,24-08-85,Self employed,01-08-18,6,1998,1,1,0,0,0,0,0,No Bureau History Available,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0yrs 0mon,0yrs 0mon,0,0


### Step 2: Handle Missing Values in `Employment.Type`

Rather than dropping rows or guessing between salaried and self-employed, we impute missing records with a distinct category `'Missing'`. This preserves all 233,154 rows and allows the model to learn if non-disclosure correlates with default.

In [2]:
print("--- Before Imputation ---")
print(df['Employment.Type'].value_counts(dropna=False))

# Impute missing values with 'Missing'
df['Employment.Type'] = df['Employment.Type'].fillna('Missing')

print("\n--- After Imputation ---")
print(df['Employment.Type'].value_counts(dropna=False))

# Verify zero missing values remain in entire dataset
remaining_nulls = df.isnull().sum().sum()
print(f"\nTotal remaining NaN values in dataset: {remaining_nulls}")
assert remaining_nulls == 0, "There are still missing values!"

--- Before Imputation ---
Employment.Type
Self employed    127635
Salaried          97858
NaN                7661
Name: count, dtype: int64

--- After Imputation ---
Employment.Type
Self employed    127635
Salaried          97858
Missing            7661
Name: count, dtype: int64

Total remaining NaN values in dataset: 0


### Step 3: Data Sanity Checks (Financial Ranges & Boundaries)

Validate that loan disbursed amounts, asset costs, and Loan-to-Value (LTV) ratios are positive and mathematically valid.

In [3]:
# Sanity check 1: Negative or zero loan amounts
invalid_disbursed = (df['disbursed_amount'] <= 0).sum()
invalid_asset = (df['asset_cost'] <= 0).sum()
invalid_ltv = ((df['ltv'] <= 0) | (df['ltv'] > 100)).sum()

print("Data Sanity Audit:")
print(f"  - Invalid Disbursed Amounts (<=0): {invalid_disbursed}")
print(f"  - Invalid Asset Costs (<=0):       {invalid_asset}")
print(f"  - Invalid LTV (<=0% or >100%):     {invalid_ltv}")
print(f"  - LTV Min: {df['ltv'].min():.2f}% | LTV Max: {df['ltv'].max():.2f}%")

Data Sanity Audit:
  - Invalid Disbursed Amounts (<=0): 0
  - Invalid Asset Costs (<=0):       0
  - Invalid LTV (<=0% or >100%):     0
  - LTV Min: 10.03% | LTV Max: 95.00%


### Step 4: Drop Non-Predictive & Leaky Identifiers

Columns like `UniqueID` (applicant database ID), `Employee_code_ID`, `supplier_id`, and `Current_pincode_ID` are high-cardinality IDs. If fed into tree models or neural networks, the model will memorize specific IDs rather than learning general credit risk patterns (overfitting).

In [4]:
# Columns to drop: arbitrary database keys and extreme high-cardinality noise IDs
cols_to_drop = ['UniqueID', 'supplier_id', 'Current_pincode_ID', 'Employee_code_ID']

print(f"Columns before dropping: {df.shape[1]}")
df_cleaned = df.drop(columns=cols_to_drop)
print(f"Columns after dropping {cols_to_drop}: {df_cleaned.shape[1]}")

Columns before dropping: 41
Columns after dropping ['UniqueID', 'supplier_id', 'Current_pincode_ID', 'Employee_code_ID']: 37


### Step 5: Verify and Export Cleaned Dataset

Save the cleaned baseline to `data/train_cleaned.csv` for Phase 3 (Feature Engineering).

In [5]:
OUTPUT_PATH = Path("../data/train_cleaned.csv") if Path("../data").exists() else Path("data/train_cleaned.csv")
df_cleaned.to_csv(OUTPUT_PATH, index=False)
print(f"Successfully exported cleaned dataset to: {OUTPUT_PATH.resolve()}")
print(f"Final Clean Shape: {df_cleaned.shape[0]:,} rows x {df_cleaned.shape[1]} columns")
df_cleaned.head(2)

Successfully exported cleaned dataset to: C:\Users\Ratul\Desktop\ml prjct\Loan\data\train_cleaned.csv
Final Clean Shape: 233,154 rows x 37 columns


,disbursed_amount,asset_cost,ltv,branch_id,manufacturer_id,Date.of.Birth,Employment.Type,DisbursalDate,State_ID,MobileNo_Avl_Flag,Aadhar_flag,PAN_flag,VoterID_flag,Driving_flag,Passport_flag,PERFORM_CNS.SCORE,PERFORM_CNS.SCORE.DESCRIPTION,PRI.NO.OF.ACCTS,PRI.ACTIVE.ACCTS,PRI.OVERDUE.ACCTS,PRI.CURRENT.BALANCE,PRI.SANCTIONED.AMOUNT,PRI.DISBURSED.AMOUNT,SEC.NO.OF.ACCTS,SEC.ACTIVE.ACCTS,SEC.OVERDUE.ACCTS,SEC.CURRENT.BALANCE,SEC.SANCTIONED.AMOUNT,SEC.DISBURSED.AMOUNT,PRIMARY.INSTAL.AMT,SEC.INSTAL.AMT,NEW.ACCTS.IN.LAST.SIX.MONTHS,DELINQUENT.ACCTS.IN.LAST.SIX.MONTHS,AVERAGE.ACCT.AGE,CREDIT.HISTORY.LENGTH,NO.OF_INQUIRIES,loan_default
0,50578,58400,89.55,67,45,01-01-84,Salaried,03-08-18,6,1,1,0,0,0,0,0,No Bureau History Available,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0yrs 0mon,0yrs 0mon,0,0
1,47145,65550,73.23,67,45,31-07-85,Self employed,26-09-18,6,1,1,0,0,0,0,598,I-Medium Risk,1,1,1,27600,50200,50200,0,0,0,0,0,0,1991,0,0,1,1yrs 11mon,1yrs 11mon,0,1
